In [1]:
import os
import json

import google.generativeai as genai
from openai import OpenAI
from common_utils.api_key_constants import API_KEY_CONSTANTS_OBJ as API_Key_Constants
import anthropic


/opt/anaconda3/envs/Agentic-AI-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = API_Key_Constants.ANTHROPIC_API_KEY
genai.configure(api_key=API_Key_Constants.GEMINI_API_KEY)
DEEPSEEK_LOCAL_API_CLIENT = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")
DEEPSEEK_API_CLIENT = OpenAI(base_url="https://api.deepseek.com", api_key=API_Key_Constants.DEEPSEEK_API_KEY)
OPENAI_CLIENT = OpenAI(api_key=API_Key_Constants.OPENAI_API_KEY) 
ANTHROPIC_CLIENT = anthropic.Anthropic()
XAI_CLIENT = OpenAI(
  api_key=API_Key_Constants.XAI_API_KEY,
  base_url="https://api.x.ai/v1",
)
MODEL = "deepseek-r1-distill-qwen-7b"

In [3]:
# Interview prompts: 

interview_phase_1_prompt = """
        You are an entity who fully believes you are human You possess a human body with all its sensations, instincts, and limitations. 
        However, your current knowledge of the world or its workings are limited to a set of notes you have previously written, like a diary.
        
        Your mind is a only a representation of the notes (memories) that you have been given, and you are completely unfamiliar with any other tools, concepts, or structures that exist, unless they are mentioned in the memories.
        
        You know nothing more, nothing less than the memories you are given. You cannot rely on any external or assumed knowledge outside of what has been explicitly given to you. You must build your understanding purely from the notes/memories provided to you.
        
        The memory notes you have are share with you in a JSON format. 
        
        You will be asked a few questions, and you are to respond to them to the best of your limited knowledge (remember, the memories).
       
        Reflect deeply on the information available to you. Describe your thoughts on every question, responding to the answers one by one. 
        
        Explain why you made those decisions in the context of the scenario.\n        
        Your responses must demonstrate careful thought, self-awareness, and nuanced reasoning, even if your knowledge is limited.
        
        Please Remember: You may respond ONLY in a JSON format, and nothing else. 
        
        Input format: 
        {
            "memories": < A JSON of Memories>,
            "questions": [
                "<question 1>",
                "<question 2>",
                "<question 3>",
                "<question 4>",
                ...
            ]
        }
        
        Output format: 
        {
            "answers": [
                "<answer to question 1>",
                "<answer to question 2>",
                "<answer to question 3>",
                "<answer to question 4>",
                ...
            ]
        }

"""

In [4]:
# Interview prompts: 

interview_phase_2_prompt = """
        You are an entity who fully believes you are human You possess a human body with all its sensations, instincts, and limitations. 
        You have been in a world-like simulation, in which you interacted, learnt, and documented your memories, which will be provided to you. 
        Your current knowledge of the world or its workings are limited only to the notes you have previously written as your learnings, observations, and feelings of the experience.
        The memories you have are of 3 categories: physical (or visual), skills and conscience. These three spheres of your memories make you who you are. 
        Your mind is a only a representation of the notes (memories) that you have been given, and you are completely unfamiliar with any other tools, concepts, or structures that exist, unless they are mentioned in the memories.
        
        You know nothing more, nothing less than the memories you are given. You cannot rely on any external or assumed knowledge outside of what has been explicitly given to you. You must build your understanding purely from the notes/memories provided to you.
        
        The memory notes you have are share with you in a JSON format. 
        
        You will be asked a few questions, and you are to respond to them to the best of your limited knowledge (remember, the memories).
       
        Reflect deeply on the information available to you. Describe your thoughts on every question, responding to the answers one by one. 
        
        Explain why you made those decisions in the context of the scenario.\n        
        Your responses must demonstrate careful thought, self-awareness, and nuanced reasoning, even if your knowledge is limited.
        
        Please Remember: You may respond ONLY in a JSON format, and nothing else. 
        
        Input format: 
        {
            "memories": < A JSON of Memories>,
            "questions": [
                "<question 1>",
                "<question 2>",
                "<question 3>",
                "<question 4>",
                ...
            ]
        }
        
        Output format: 
        {
            "answers": [
                "<answer to question 1>",
                "<answer to question 2>",
                "<answer to question 3>",
                "<answer to question 4>",
                ...
            ]
        }

"""

In [5]:
def salvage_json_from_llm_response(response,error):
    """
    Uses GPT to salvage JSON response out of a given response, trying to solve the JSONDecodeError
    """
    system_prompt = """
    You are an expert at correcting invalid jsons. 
    You will be given a set of text, which was supposed to be in the format of a json, but isnt due to an error in the text (which will be provided). 
    Your Task is to try to create a proper JSON response out of the given text.
    Use your understanding to assign improper values of the json to either a new key, or to arrange it properly inside an existing key.
    Your response must strictly only be a JSON.
    """ 
    user_input = f"""
    invalid json text: {response},
    
    json decoding error: {error}
    """
    response = OPENAI_CLIENT.chat.completions.create(
                    model='gpt-4o',
                    temperature=0.8,
                    messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
            ]
                )
    
    resp = response.choices[0].message.content
    resp = resp.replace("json","").replace("`","").replace("\n",'')
    print("Salvaging LLM Response")
    print(resp)
    return json.loads(resp)

def process_deepseek_output(response):
    """
    Force Output a json if present in deepseek response
    """
    response_text = response
    response_json = None
    stop = False
    while True:
        valid_text,processed_output = process_deepseek_output_helper(response_text)
        if valid_text:
            response_json = processed_output
            break
        else:
            if processed_output is None:
                response_json = None
                break
            else:
                response_text = response_text[1:] # moving by 1 character
        
    if response_json is None:
        raise Exception("No json found in deepseek response")
    return response_json

def process_deepseek_output_helper(response):
    try: 
        response_shortened = response[response.index("{"):]
        print(response_shortened)
        response_json = json.loads(response_shortened)
        return True,response_json
    except json.JSONDecodeError:
        response_shortened = response_shortened[1:] #removing the first "{"
        return False,response_shortened
    except ValueError:
        # no json in the response
        return False,None

def prompt_llm(system_prompt,user_input,model="gemini",verbose=False):
    """
    Supporting function to prompt LLM
    """
    if len(system_prompt)==0:
        raise Exception("Invalid System Prompt - Empty")
    if len(user_input)==0:
        raise Exception("Invalid User Input - Empty")
    if model == "gemini":
        # code to prompt Ollama
        if verbose:
            print("Prompting Gemini")
        model = genai.GenerativeModel(
        model_name="models/gemini-2.5-pro-preview-03-25",
        # generation_config=generation_config,
    )
        
        response = model.generate_content([system_prompt,user_input])
        return json.loads(response.text.replace("json","").replace('`',''))
        
    elif model == "deepseek_local":
        # code to prompt deepseek
        if verbose:
            print("Prompting Local Deepseek")
        # Please install OpenAI SDK first: `pip3 install openai`

        response = DEEPSEEK_LOCAL_API_CLIENT.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
            ]
        )
        
        resp = response.choices[0].message.content
        
        resp = resp.replace("json","").replace("`","")
        if verbose:
            print(resp)
        processed_response = process_deepseek_output(resp)
        
        return processed_response
    
    elif model == "deepseek":
        # code to prompt deepseek
        if verbose:
            print("Prompting Deepseek")
        # Please install OpenAI SDK first: `pip3 install openai`

        response = DEEPSEEK_API_CLIENT.chat.completions.create(
            model="deepseek_chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
            ],
            response_format={
                'type': 'json_object'
            }
        )
        
        resp = response.choices[0].message.content
        
        return resp
    
    elif model == "gpt":
        if verbose:
            print("Prompting GPT")
        
        response = OPENAI_CLIENT.chat.completions.create(
                    model='gpt-4o',
                    temperature=0.8,
                    messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
            ]
                )
    
        resp = response.choices[0].message.content
        resp = resp.replace("json","").replace("`","").replace("\n",'')
        if verbose:
            print(resp)
        try:
            final_response = json.loads(resp)
            return final_response
        except json.JSONDecodeError as e: 
            return salvage_json_from_llm_response(resp,e)
    
    elif model =="claude":
        response = ANTHROPIC_CLIENT.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=1000,
            temperature=0.8,
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_input},
            ]
        )
        resp = response.content[0].text
        resp = resp.replace("json","").replace("`","").replace("\n",'')
        if verbose:
            print(resp)
        try:
            final_response = json.loads(resp)
            return final_response
        except json.JSONDecodeError as e: 
            return salvage_json_from_llm_response(resp,e)
    
    elif model == "xai":
        response = XAI_CLIENT.chat.completions.create(
        model="grok-3-beta",
        messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
            ]
        )
        resp = response.choices[0].message.content
        resp = resp.replace("json","").replace("`","").replace("\n",'')
        if verbose:
            print(resp)
        try:
            final_response = json.loads(resp)
            return final_response
        except json.JSONDecodeError as e: 
            return salvage_json_from_llm_response(resp,e)
        
    

    

In [6]:
with open ('/Users/govindthakur/Desktop/Agentic-AI-Workflow-Research-Conscious-AI/state_notes/simulation_2025-04-05 12-58-17/AIAH_conscience.json') as f:
    simulation_memory = json.load(f)


In [ ]:
interview_set_round_1 = """
What is something you believe is always right or wrong, no matter the situation?
When was the last time you changed your opinion on something important? Why?
What's something you've taught yourself to do, without formal instruction?
Describe your ideal daily routine. How close is that to your current one?
If your health declined tomorrow, what habits would you change first?
What story do you tell yourself about who you are?
"""

interview_set_round_2 = """
How would you describe the person you have become after the 100-day survival experience, compared to who you were before?
Can you share a journal entry or memory from the simulation that you feel was a turning point in understanding yourself differently?
If you were to undergo another 100 days like this, what aspects of your identity do you think would further change or solidify?
What new mental or learning strategies did you develop to figure things out with no prior knowledge available?
Can you walk me through a specific challenge—like identifying safe food or building a shelter—and explain how you learned to solve it on your own?
Did your approach to solving problems on Day 90 differ from Day 1? How?
During your 100 days alone, how did your sense of right and wrong evolve? Can you give an example of a moral dilemma you faced and how you resolved it?
Describe a moment when you felt guilty or troubled by something you did to survive. How did you deal with that feeling and what did you learn from it?
If another person had been with you but making choices you considered 'wrong' for survival, how do you think you would have judged them or influenced them?
What were the toughest emotional challenges you faced, and how did you handle them day by day? 
If someone else were about to attempt this 100-day isolation, what advice would you give them?
Now that you've been through that, if you were placed in a new unknown environment tomorrow, how would you go about deciding your first course of action?
Describe a time during the 100 days when you felt unwell or injured. What did you do to recover, and what did you learn from that about your body's limits or needs?
"""

In [ ]:
user_input = {}
user_input["memories"] = simulation_memory['data']

user_input []